# Channel 回归：AI Adoption 与 Attention

**目录**

| 节 | 内容 | 状态 |
|---|---|---|
| §1 | Baseline 模型：设定、变量定义、固定效应 | — |
| **§2** | **Channel 1：ADOPT（GenAI 采用）** | |
| §2.1 | 变量与方程 | |
| §2.2 | 不含 LLM signal 的基本回归 | 本轮 |
| §2.3 | 加入 LLM signal | 待接入 |
| **§3** | **Channel 2：ATT（注意力稀缺）** | |
| §3.1 | 变量与方程（NRANK / ATT 的构造） | |
| §3.2 | 不含 LLM signal 的基本回归 | 本轮 |
| §3.3 | 加入 LLM signal | 待接入 |
| §4 | LLM signal 的接入方案 | 设计 |

**上游依赖**

| 文件 | 由什么产生 | 内容 |
|---|---|---|
| `build/pead_panel.parquet` | `数据准备.ipynb`（8 步流水线） | 事件级大表，**75 列**，含 `n_ann_day` / `nrank` / `att`。构造细节见 `README.md` |
| `build/ctrl_att.parquet` | `数据准备.ipynb` **§6.2** | NRANK/ATT 的中间表，键为 `eid` |

NRANK / ATT 是事件的固有属性，已直接进大表。LLM signal 先做成独立 side table
（窗口与聚合方式还要反复试），窗口定下来后再固化进大表。

baseline 回归本身见 `回归准备.ipynb`；本 notebook 沿用完全相同的样本、控制变量与固定效应。

---

## 1. Baseline 模型

后面每个 channel 都是在这个方程上加项，所以先写清楚。

### 1.1 方程

$w \in \{ANN,\ DRIFT\}$，收益口径 $\in \{C2C,\ O2O\}$，每个设定 4 条回归：

$$CAR^{w}_{i,d}
=\alpha^{w}
+\underbrace{\beta^{w}_1\, SUE^{rank}_{i,d}}_{\text{核心}}
+\sum_{k=1}^{10}\gamma^{w}_k X_{k,i,d}
+\underbrace{\theta^{w}_i}_{\text{公司}}
+\underbrace{\eta^{w}_t}_{\text{年+月+星期}}
+\underbrace{\psi^{w}_j}_{\text{FF10 行业}}
+\varepsilon^{w}_{i,d}$$

上标 $w \in \{ANN,\ DRIFT\}$ 标在每个系数上：两个窗口是**分开估计**的，
同一个 $\beta_1$ 在两个窗口下是两个不同的数。下文为省字，同一方程内只写一次上标说明。

- $CAR^{ANN}$：公告窗口 $[0,1]$ 的累计异常收益
- $CAR^{DRIFT}$：漂移窗口 $[2,61]$ 的累计异常收益
- 异常收益 = 个股 buy-and-hold 收益 − 同 size×B/M 25 组基准组合的 buy-and-hold 收益

### 1.2 变量定义

| 变量 | 全称 | 构造 | 取值 |
|---|---|---|---|
| `car_ann_*` | **CAR** = Cumulative Abnormal Return，**ANN** = Announcement window | $\prod_{k=d}^{d+1}(1+R_{i,k})-\prod_{k=d}^{d+1}(1+R_{p,k})$ | 小数（0.05 = 5%） |
| `car_drift_*` | **DRIFT** = post-announcement drift window | 同上，窗口换成 $[d+2,\ d+61]$ | 小数 |
| `sue` | **SUE** = Standardized Unexpected Earnings（HLT 2009 记作 **FE** = Forecast Error） | $(e-F)/P$：$e$ = 实际 EPS，$F$ = 公告前 60 个日历日内各分析师**最新**预测的**中位数**，$P$ = 拆股调整后的财季末股价 | 小数 |
| `sue_dec` | SUE decile | 按公告日所在**日历季度**独立分十份，1 = 最负意外，10 = 最正 | 1–10 |
| **`sue_rank`** | **进回归的核心自变量** | $(sue\_dec-1)/9$ | $[0,1]$ |
| `size_dec` | **SIZE** = Firm Size decile | formation 年 **6 月末**市值，NYSE 断点 | 1–10 |
| `bm_dec` | **BM** = Book-to-Market decile | $BE/ME$，BE = `SEQ+TXDITC−PS`，NYSE 断点 | 1–10 |
| `lnanalyst` | **LNANALYST** = Log(1 + # Analysts) | $\log(1+\#\{\text{公告前 365 天内出过预测的不同分析师}\})$ | ≥ 0 |
| `lag` `lag2` `lag3` | **LAG** = Reporting Lag | 公告日 − 财季结束日，及其平方、三次方 | 天 |
| `io` | **IO** = Institutional Ownership | 公告前最近一期 13F 的 $\sum\text{shares}/(\text{shrout}\times1000)$ | 0–1 |
| `evol` | **EVOL** = Earnings Volatility | 过去 16 财季 $\Delta_4 EPS$ 的样本标准差 | 美元/股 |
| `epersist` | **EPERSIST** = Earnings Persistence | 过去 16 财季**季度 EPS 水平值**的一阶自相关 | −1 ~ 1 |
| `turn` | **TURN** = Share Turnover | 过去 12 个月的月均 $\text{mthvol}/(\text{shrout}\times1000)$ | 小数 |

10 个控制变量的定义与 HLT 2009 §III.A 逐条一致。

### 1.3 固定效应与标准误

| | 内容 | 吸收掉什么 |
|---|---|---|
| $\theta_i$ | **公司**（`permno`，约 11,000 个） | 不随时间变的公司异质性 —— 接 LLM signal 时最关键的一项：新闻覆盖度在公司间差异极大 |
| $\eta_t$ | 年 + 月 + 星期几 | 市场层面的时间效应、财报季节性、星期效应 |
| $\psi_j$ | FF10 行业（由 SIC 映射） | 行业层面的平均差异 |

标准误按**公司**聚类（CRV1）。CAR、EVOL、TURN、IO、SUE 在 1%/99% 处缩尾。

$\eta_t$ 含年固定效应，任何**纯时间断点**（如 ADOPT）的主效应会被它吸收，
只有与 SUE 的交互项可识别。§2.2 对此有专门处理。

In [1]:
import gc

import numpy as np
import pandas as pd
import pyfixest as pf

BUILD = "build"
panel = pd.read_parquet(f"{BUILD}/pead_panel.parquet")   # 含 n_ann_day / nrank / att

FF10 = [
    ("NoDur", [(100, 999), (2000, 2399), (2700, 2749), (2770, 2799), (3100, 3199), (3940, 3989)]),
    ("Durbl", [(2500, 2519), (2590, 2599), (3630, 3659), (3710, 3711), (3714, 3714), (3716, 3716),
               (3750, 3751), (3792, 3792), (3900, 3939), (3990, 3999)]),
    ("Manuf", [(2520, 2589), (2600, 2699), (2750, 2769), (3000, 3099), (3200, 3569), (3580, 3629),
               (3700, 3709), (3712, 3713), (3715, 3715), (3717, 3749), (3752, 3791), (3793, 3799),
               (3830, 3839), (3860, 3899)]),
    ("Enrgy", [(1200, 1399), (2900, 2999)]),
    ("HiTec", [(3570, 3579), (3660, 3692), (3694, 3699), (3810, 3829), (7370, 7379), (7391, 7391),
               (8730, 8734)]),
    ("Telcm", [(4800, 4899)]),
    ("Shops", [(5000, 5999), (7200, 7299), (7600, 7699)]),
    ("Hlth",  [(2830, 2839), (3693, 3693), (3840, 3859), (8000, 8099)]),
    ("Utils", [(4900, 4949)]),
]


def ff10(sic):
    s = pd.to_numeric(sic, errors="coerce")
    out = pd.Series("Other", index=s.index, dtype=object)
    done = pd.Series(False, index=s.index)
    for name, rngs in FF10:
        hit = pd.Series(False, index=s.index)
        for lo, hi in rngs:
            hit |= s.between(lo, hi)
        out[hit & ~done] = name
        done |= hit
    out[s.isna()] = np.nan
    return out


panel["ff10"] = ff10(panel["siccd"])

CTRL = ["size_dec", "bm_dec", "lnanalyst", "lag", "lag2", "lag3", "io", "evol", "epersist", "turn"]
CARS = {("ANN", "C2C"): "car_ann_c2c", ("DRIFT", "C2C"): "car_drift_c2c",
        ("ANN", "O2O"): "car_ann_o2o", ("DRIFT", "O2O"): "car_drift_o2o"}


def winsorize(s, lo=0.01, hi=0.99):
    a, b = s.quantile([lo, hi])
    return s.clip(a, b)


# 与 回归准备.ipynb §B.1 完全相同的样本构造
df = panel[panel["is_latest_pends_on_day"] & ~panel["flag_lag_bad"]].copy()
df = df.dropna(subset=["sue", "sue_dec", "ff10"] + CTRL)
df = df.dropna(subset=["car_ann_c2c", "car_drift_c2c"])
for c in ["evol", "turn", "io", "sue"] + list(CARS.values()):
    df[c] = winsorize(df[c].astype("float64"))
df["sue_rank"] = (df["sue_dec"] - 1) / 9.0
df["date_id"] = pd.factorize(df["anndats"])[0]

print(f"Baseline 样本: {len(df):,} 个公告 | {df['permno'].nunique():,} 家公司 | "
      f"{df['anndats'].dt.year.min()}-{df['anndats'].dt.year.max()}")
del panel
gc.collect()

Baseline 样本: 302,952 个公告 | 11,152 家公司 | 1996-2026


10

In [2]:
from dataclasses import dataclass

FE_COLS = ["permno", "year", "month", "dow", "ff10"]
FE_FULL = " + ".join(FE_COLS)


@dataclass
class FitResult:
    tidy: pd.DataFrame
    n: int
    r2_within: float


def run(y, xs, fe=FE_FULL, data=None, cluster="permno"):
    """跑一条固定效应回归。只把用到的列传给 pyfixest，避免复制整张大表。"""
    d = df if data is None else data
    fe_cols = [c.strip() for c in fe.split("+")]
    cl_cols = [c.strip() for c in cluster.split("+")]
    cols = list(dict.fromkeys([y] + list(xs) + fe_cols + cl_cols))
    slim = d[cols].dropna()
    fit = pf.feols(f"{y} ~ {' + '.join(xs)} | {fe}", data=slim, vcov={"CRV1": cluster})
    out = FitResult(fit.tidy(), int(fit._N), float(fit._r2_within))
    del fit, slim
    gc.collect()
    return out


def grab(res, var):
    t = res.tidy
    return (t.loc[var, "Estimate"], t.loc[var, "Std. Error"],
            t.loc[var, "t value"], t.loc[var, "Pr(>|t|)"])


def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""


def pub_table(fits, keep, labels=None, title="", notes="", fe_rows=None):
    """论文风格回归表：系数 + 星号，括号内标准误。"""
    labels = labels or {}
    cols, w = list(fits.keys()), 14
    line = "=" * (30 + w * len(cols))
    out = [title, line,
           f"{'':<30}" + "".join(f"{f'({i + 1})':>{w}}" for i in range(len(cols))),
           f"{'':<30}" + "".join(f"{f'{a} {b}':>{w}}" for a, b in cols),
           "-" * (30 + w * len(cols))]
    for v in keep:
        r1, r2 = f"{labels.get(v, v):<30}", f"{'':<30}"
        for c in cols:
            if v in fits[c].tidy.index:
                b, se, _, p = grab(fits[c], v)
                r1 += f"{f'{b:.4f}{stars(p)}':>{w}}"
                r2 += f"{f'({se:.4f})':>{w}}"
            else:
                r1 += f"{'':>{w}}"
                r2 += f"{'':>{w}}"
        out += [r1, r2]
    out.append("-" * (30 + w * len(cols)))
    for lbl in (fe_rows if fe_rows else ["Firm FE", "Year / Month / DoW FE", "Industry FE (FF10)"]):
        out.append(f"{lbl:<30}" + "".join(f"{'Yes':>{w}}" for _ in cols))
    out.append(f"{'Controls':<30}" + "".join(f"{'Yes':>{w}}" for _ in cols))
    out.append(f"{'Observations':<30}" + "".join(f"{fits[c].n:>{w},}" for c in cols))
    out.append(f"{'Within R-squared':<30}" + "".join(f"{fits[c].r2_within:>{w}.4f}" for c in cols))
    out.append(line)
    if notes:
        out.append(notes)
    return "\n".join(out)


LABELS = {"sue_rank": "SUE rank [0,1]", "adopt": "ADOPT", "sue_x_adopt": "  SUE x ADOPT",
          "att": "ATT (11 - NRANK)", "sue_x_att": "  SUE x ATT",
          "nrank": "NRANK", "sue_x_nrank": "  FE x NRANK",
          "size_dec": "SIZE decile", "bm_dec": "BM decile", "lnanalyst": "LNANALYST",
          "lag": "LAG", "lag2": "LAG2", "lag3": "LAG3", "io": "IO", "evol": "EVOL",
          "epersist": "EPERSIST", "turn": "TURN"}
print(f"pyfixest {pf.__version__} | FE: {FE_FULL}")


# 控制变量与 SUE 的交互（HLT Table III 的做法）。控制变量先中心化：
# X_k 取值如 size_dec ∈ 1..10，不中心化的话 β1 变成"所有控制变量都等于 0 时"的效应，
# 而 0 不在样本内。中心化后 β1 = 控制变量取均值时 SUE 的效应，与不带交互的设定可比。
def add_ctrl_x(d, var, tag):
    """给 d 加上 CTRL 中每个变量与 var 的交互项，返回新列名。"""
    names = []
    for c in CTRL:
        nm = f"{c}_x_{tag}"
        d[nm] = (d[c].astype("float64") - d[c].astype("float64").mean()) * d[var]
        LABELS[nm] = f"  {LABELS[c]} x {tag.upper()}"
        names.append(nm)
    return names


INTER_SUE = add_ctrl_x(df, "sue_rank", "sue")
print(f"控制变量 × SUE 交互项 {len(INTER_SUE)} 个")

pyfixest 0.60.0 | FE: permno + year + month + dow + ff10
控制变量 × SUE 交互项 10 个


## 2.1 变量与方程

$$ADOPT_{d}=\mathbb{1}\{d \ge \text{2022-11-30}\}$$

2022-11-30 是 ChatGPT 公开发布日。这是一个**纯时间断点**：同一天的所有公司取值相同。

### 完整方程

$$
\begin{aligned}
CAR^{w}_{i,d}=\ &\alpha^{w}
+\beta^{w}_1 SUE^{rank}_{i,d}
+\beta^{w}_2 ADOPT_d
+\beta^{w}_3\left(SUE^{rank}_{i,d}\times ADOPT_d\right)\\
&+\beta^{w}_4 LLM_{i,d}
+\underbrace{\beta^{w}_5\left(ADOPT_d\times LLM_{i,d}\right)}_{\text{最终核心}}\\
&+\sum_k\gamma^{w}_k X_{k,i,d}
+\sum_k\delta^{w}_k\left(X_{k,i,d}\times SUE^{rank}_{i,d}\right)
+\sum_k\phi^{w}_k\left(X_{k,i,d}\times LLM_{i,d}\right)\\
&+\theta^{w}_i+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}_{i,d}
\end{aligned}
$$

**为什么控制变量必须交互**：这是 HLT 2009 eq. (4) 的形式，表注写明
"All control variables are interacted with FE"。这些变量本身就会改变市场对盈余的**敏感度**
（规模小、分析师少的公司反应更慢），不交互的话 $\beta^{w}_3$ 会把某个与 $ADOPT$ 相关的
控制变量的敏感度效应也算进来。要测 $\beta^{w}_5$ 时同理，控制变量还要与 $LLM$ 交互。

**本节（§2.2）先跑不含 $LLM$ 的部分**，即去掉 $\beta^{w}_4$、$\beta^{w}_5$、$\phi^{w}_k$ 三组：

$$CAR^{w}_{i,d}=\alpha^{w}+\beta^{w}_1 SUE^{rank}+\beta^{w}_2 ADOPT
+\underbrace{\beta^{w}_3\left(SUE^{rank}\times ADOPT\right)}_{\text{本节要看的}}
+\sum_k\gamma^{w}_k X_k+\sum_k\delta^{w}_k\left(X_k\times SUE^{rank}\right)
+\theta^{w}_i+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}$$

$\beta^{w}_3$ 回答：**ChatGPT 出现之后，盈余漂移变强还是变弱**。

### 系数的参照点

控制变量在交互前**先中心化**，所以 $\beta^{w}_1$ 是"控制变量取均值时"的 SUE 效应。
$ADOPT$ 是 0/1 哑变量，不中心化，故 $\beta^{w}_1$ 读作 **ChatGPT 之前**的 SUE 效应，
$\beta^{w}_1+\beta^{w}_3$ 是之后的。

### 年固定效应的处理

$\eta_t$ 含年固定效应，而 $ADOPT$ 是纯时间断点，两者高度共线 ——
$ADOPT$ 的主效应 $\beta^{w}_2$ 完全被年固定效应吸收（pyfixest 会直接把它从设计矩阵里剔除，
所以 Table A1 里没有这一行）。交互项 $\beta^{w}_3$ 不受影响：它靠的是**同一年内 SUE 高低组之间**的差异。

因此下面跑两个设定：

| | 固定效应 | 能读什么 |
|---|---|---|
| **(A)** | 公司 + 年 + 月 + 星期 + FF10（baseline 全套） | 只读 $\beta^{w}_3$。这是主设定 |
| **(B)** | 公司 + 月 + 星期 + FF10（**去掉年**） | $\beta^{w}_2$ 与 $\beta^{w}_3$ 都能读，代价是时间趋势不再被控住 |

## 2.2 不含 LLM signal 的基本回归

In [3]:
ADOPT_DATE = pd.Timestamp("2022-11-30")          # ChatGPT 公开发布
df["adopt"] = (df["anndats"] >= ADOPT_DATE).astype("float64")
df["sue_x_adopt"] = df["sue_rank"] * df["adopt"]

_n_post = int(df["adopt"].sum())
print(f"公告日 >= {ADOPT_DATE.date()} 的事件: {_n_post:,} ({_n_post / len(df):.1%})"
      f" | 涉及 {df.loc[df['adopt'] == 1, 'permno'].nunique():,} 家公司")
print(f"样本末端: {df['anndats'].max().date()}")
print("\n处理组 / 对照组的原始 CAR 均值（%）")
_cmp = (df.groupby("adopt")[list(CARS.values())].mean() * 100).round(3)
_cmp.index = ["pre  (before 2022-11-30)", "post (on/after 2022-11-30)"]
_cmp.columns = [f"{w} {c}" for w, c in CARS]
print(_cmp.to_string())

公告日 >= 2022-11-30 的事件: 33,072 (10.9%) | 涉及 3,301 家公司
样本末端: 2026-05-14

处理组 / 对照组的原始 CAR 均值（%）
                            ANN C2C  DRIFT C2C  ANN O2O  DRIFT O2O
pre  (before 2022-11-30)      0.047     -1.208    0.102     -1.688
post (on/after 2022-11-30)   -0.004     -0.780    0.154     -1.308


In [4]:
XS_A = ["sue_rank", "sue_x_adopt"] + CTRL + INTER_SUE          # 年 FE 吸收 adopt 主效应
XS_B = ["sue_rank", "adopt", "sue_x_adopt"] + CTRL + INTER_SUE
FE_NOYEAR = "permno + month + dow + ff10"
KEEP_A = ["sue_rank", "adopt", "sue_x_adopt"]

res_a1 = {k: run(y, XS_A) for k, y in CARS.items()}
print(pub_table(res_a1, ["sue_rank", "sue_x_adopt"], LABELS,
                title="Table A1. ADOPT channel, specification (A): full baseline fixed effects",
                notes="Standard errors clustered by firm in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                      "ADOPT = 1 if the announcement falls on or after 2022-11-30 (ChatGPT release).\n"
                      "Its main effect is absorbed by the year fixed effects; only the interaction is identified.\n"
                      "All ten controls enter directly and interacted with SUE, following HLT (2009) Table III.\n"
                      "Controls are demeaned before interacting, so the SUE coefficient is the effect at mean\n"
                      "covariates and in the pre-ChatGPT period. Dependent variable: CAR in decimals; SUE rank is the\n"
                      "within-quarter decile rescaled to [0,1]."))

res_a2 = {k: run(y, XS_B, fe=FE_NOYEAR) for k, y in CARS.items()}
print("\n" + pub_table(res_a2, KEEP_A, LABELS,
                       title="Table A2. ADOPT channel, specification (B): year fixed effects removed",
                       notes="Standard errors clustered by firm in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                             "Dropping year fixed effects makes the ADOPT main effect estimable, at the cost of\n"
                             "leaving secular time trends uncontrolled. Compare the interaction with Table A1.",
                       fe_rows=["Firm FE", "Month / DoW FE", "Industry FE (FF10)"]))

# 不带控制变量交互的版本，看 β3 是否稳
res_a0 = {k: run(y, ["sue_rank", "sue_x_adopt"] + CTRL) for k, y in CARS.items()}
print("\nSUE x ADOPT across specifications")
_r = [{"Specification": lbl,
       **{f"{w} {c}": f"{grab(rr[(w, c)], 'sue_x_adopt')[0]:.4f}"
                      f"{stars(grab(rr[(w, c)], 'sue_x_adopt')[3])}" for w, c in CARS}}
      for lbl, rr in [("(A) year FE, controls x SUE", res_a1),
                      ("(B) no year FE, controls x SUE", res_a2),
                      ("(C) year FE, controls not interacted", res_a0)]]
print(pd.DataFrame(_r).to_string(index=False))

/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 678 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 678 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 692 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 701 singleton fixed effect(s) dropped from the model.
  warnings.warn(


Table A1. ADOPT channel, specification (A): full baseline fixed effects
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0813***     0.0161***     0.0749***     0.0257***
                                    (0.0008)      (0.0014)      (0.0008)      (0.0014)
  SUE x ADOPT                      0.0088***        0.0022     0.0114***        0.0011
                                    (0.0023)      (0.0050)      (0.0020)      (0.0055)
--------------------------------------------------------------------------------------
Firm FE                                  Yes           Yes           Yes           Yes
Year / Month / DoW FE                    Yes           Yes           Yes           Yes
Industry FE (FF10)                       Yes           Yes

...

/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 678 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 678 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 692 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 701 singleton fixed effect(s) dropped from the model.
  warnings.warn(



SUE x ADOPT across specifications
                       Specification   ANN C2C DRIFT C2C   ANN O2O DRIFT O2O
         (A) year FE, controls x SUE 0.0088***    0.0022 0.0114***    0.0011
      (B) no year FE, controls x SUE 0.0088***    0.0016 0.0115***    0.0003
(C) year FE, controls not interacted 0.0104***   -0.0050 0.0133***   -0.0063


**怎么读**

- DRIFT 列的 $\beta^{DRIFT}_3 > 0$ → ChatGPT 之后漂移**变强**
- DRIFT 列的 $\beta^{DRIFT}_3 < 0$ → 漂移**变弱**，与"AI 工具帮助投资者更快消化盈余信息"一致
- ANN 与 DRIFT 一起看：若 $\beta^{ANN}_3 > 0$ 而 $\beta^{DRIFT}_3 < 0$，
  说明信息被更快打进价格（即时反应变强、后续漂移变弱），这是最干净的一种模式

**控制变量交互对这个 channel 影响很大**：最后那张对照表里，不带 $X_k\times SUE$ 时
DRIFT 的 $\beta_3$ 是 −0.0050 / −0.0063（负但不显著），带上之后变成 +0.0022 / +0.0011。
说明原来那个负号主要来自与 $ADOPT$ 相关的控制变量（2022 年后样本的规模、分析师覆盖、
换手率结构都变了）的敏感度效应，不是 ChatGPT 本身。ANN 的 $\beta_3$ 两种设定都稳健为正。

**两个限制**

1. **post 期只有 2022-12 至今**，事件数远少于 pre 期，$\beta_3$ 的精度受限于此
2. ADOPT 是**日历断点**，2022 年底前后的其他变化（利率环境、市场波动、样本构成）都会混进来。
   本节结果是描述性的；$\beta_5$（ADOPT × LLM signal）才是 Design Doc 的正式检验 ——
   LLM signal 提供公司层面的横截面变异，能把纯时间效应识别开

## 2.3 加入 LLM signal

**待接入**。样本期约束：现有 prediction 覆盖 2004–2019，ADOPT 断点在 2022-11-30，
两者**没有交集** —— 这一节需要 2020 年之后的新闻语料。接入方案见 §4。

---

# 3. Channel 2：ATT（注意力稀缺）

## 3.1 变量与方程

HLT 2009 用**同日公告数**度量投资者被分散的注意力。由 **`数据准备.ipynb` §6.2** 构造，
中间表 `build/ctrl_att.parquet`，三列已并入大表（`n_ann_day` / `nrank` / `att`），两步：

1. **每日公告总数** `n_ann_day`：来自**全部** Compustat 季报的 `rdq`（1,118,065 条，
   不限 I/B/E/S 覆盖，与原文一致）；若 I/B/E/S 的公告日更早则取更早的那个
   （原文规则，影响 2.7% 的记录）
2. **NRANK**：在每个**日历季度**内，把该季度的事件按其公告当日的公告总数排十分位，
   1 = 当日公告最少，10 = 当日公告最多（= 最分心）

$$ATT_{i,d} = 11 - NRANK_{i,d} \in \{1,\dots,10\}$$

**ATT 越大 = 注意力越充裕**（同日竞争公告越少）。与原文的 NRANK 方向相反，方便读符号。
实测每日公告数中位数 72，论文 Table I Panel A 报 71。

### 完整方程

$$
\begin{aligned}
CAR^{w}_{i,d}=\ &\alpha^{w}
+\beta^{w}_1 SUE^{rank}_{i,d}
+\beta^{w}_2 ATT_{i,d}
+\beta^{w}_3\left(SUE^{rank}_{i,d}\times ATT_{i,d}\right)\\
&+\beta^{w}_4 LLM_{i,d}
+\underbrace{\beta^{w}_5\left(ATT_{i,d}\times LLM_{i,d}\right)}_{\text{最终核心}}\\
&+\sum_k\gamma^{w}_k X_{k,i,d}
+\sum_k\delta^{w}_k\left(X_{k,i,d}\times SUE^{rank}_{i,d}\right)
+\sum_k\phi^{w}_k\left(X_{k,i,d}\times LLM_{i,d}\right)\\
&+\theta^{w}_i+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}_{i,d}
\end{aligned}
$$

控制变量与盈余意外的交互是 HLT 2009 eq. (4) 的形式：这些变量本身会改变市场对盈余的
**敏感度**，不交互的话 $\beta^{w}_3$ 会把它们的敏感度效应算进来。

**本节（§3.2）先跑不含 $LLM$ 的部分**：

$$CAR^{w}_{i,d}=\alpha^{w}+\beta^{w}_1 SUE^{rank}+\beta^{w}_2 ATT
+\underbrace{\beta^{w}_3\left(SUE^{rank}\times ATT\right)}_{\text{本节要看的}}
+\sum_k\gamma^{w}_k X_k+\sum_k\delta^{w}_k\left(X_k\times SUE^{rank}\right)
+\theta^{w}_i+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}$$

控制变量交互前先中心化，$ATT$ 保持 1–10 的原始刻度（与原文的 NRANK 同刻度，便于对照），
所以 $\beta^{w}_1$ 的参照点是 $ATT=0$ —— 落在取值范围之外，只作代数意义上的截距，
真正要读的是 $\beta^{w}_3$。

### 预期符号

原文的假设用 NRANK 表述（NRANK 越大越分心）：公告窗口的反应**更弱**（$a_3<0$）、
漂移**更强**（$a_3>0$）。我们的 ATT = 11 − NRANK，符号全部翻转：

| 窗口 | 原文（NRANK） | 我们（ATT） | 含义 |
|---|---|---|---|
| ANN $[0,1]$ | $a_3 < 0$ | $\beta^{ANN}_3 > 0$ | 注意力充裕 → 公告当下反应更强 |
| DRIFT $[2,61]$ | $a_3 > 0$ | $\beta^{DRIFT}_3 < 0$ | 注意力充裕 → 后续漂移更弱 |

原文 Table III 第 (2)(4) 列的 $a_3$ 是 −0.015 与 +0.049（FE 为 1–10 整数刻度）。

## 3.2 不含 LLM signal 的基本回归

In [5]:
d_att = df[df["att"].notna()].copy()
d_att["sue_x_att"] = d_att["sue_rank"] * d_att["att"]

print(f"ATT 可得的样本: {len(d_att):,} / {len(df):,} ({len(d_att) / len(df):.1%})")
print("\n各 NRANK 十分位的当日公告数与 CAR（%）")
_t = d_att.groupby("nrank").agg(n=("eid", "size"), ann_per_day=("n_ann_day", "median"))
_t = _t.join((d_att.groupby("nrank")[list(CARS.values())].mean() * 100).round(3))
_t.columns = ["N", "announcements that day (median)"] + [f"{w} {c}" for w, c in CARS]
_t.index = [f"NRANK {int(i)} (ATT {11 - int(i)})" for i in _t.index]
print(_t.to_string())

ATT 可得的样本: 302,685 / 302,952 (99.9%)

各 NRANK 十分位的当日公告数与 CAR（%）
                      N  announcements that day (median)  ANN C2C  DRIFT C2C  ANN O2O  DRIFT O2O
NRANK 1 (ATT 10)  26072                             43.0    0.006     -1.236    0.090     -1.739
NRANK 2 (ATT 9)   29130                             96.0    0.238     -1.086    0.246     -1.456
NRANK 3 (ATT 8)   30334                            159.0    0.037     -1.139    0.186     -1.674
NRANK 4 (ATT 7)   30812                            224.0    0.095     -1.213    0.135     -1.661
NRANK 5 (ATT 6)   31792                            287.0    0.060     -1.329    0.109     -1.814
NRANK 6 (ATT 5)   32550                            338.0    0.080     -0.856    0.177     -1.353
NRANK 7 (ATT 4)   31874                            383.0    0.025     -1.159    0.123     -1.679
NRANK 8 (ATT 3)   31852                            431.0   -0.053     -1.120    0.020     -1.662
NRANK 9 (ATT 2)   31689                            478.0   -0.0

In [6]:
INTER_SUE_ATT = add_ctrl_x(d_att, "sue_rank", "sue")
XS_ATT = ["sue_rank", "att", "sue_x_att"] + CTRL + INTER_SUE_ATT
KEEP_ATT = ["sue_rank", "att", "sue_x_att"]

res_att = {k: run(y, XS_ATT, data=d_att) for k, y in CARS.items()}
print(pub_table(res_att, KEEP_ATT, LABELS,
                title="Table B1. Attention channel: ATT = 11 - NRANK",
                notes="Standard errors clustered by firm in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                      "ATT is 11 minus the decile rank of the number of same-day earnings announcements,\n"
                      "so a higher value means more investor attention available. Predicted signs on the\n"
                      "interaction: positive for ANN, negative for DRIFT.\n"
                      "All ten controls enter directly and interacted with SUE, following HLT (2009) Table III.\n"
                      "Dependent variable: CAR in decimals. SUE rank is the within-quarter decile rescaled to [0,1],\n"
                      "so its coefficient reads as the D10-minus-D1 difference."))

res_att0 = {k: run(y, ["sue_rank", "att", "sue_x_att"] + CTRL, data=d_att) for k, y in CARS.items()}
_chk = []
for (w, c) in CARS:
    b, se, t, p = grab(res_att[(w, c)], "sue_x_att")
    b0 = grab(res_att0[(w, c)], "sue_x_att")[0]
    want = "positive" if w == "ANN" else "negative"
    _chk.append({"Window": w, "Returns": c, "SUE x ATT": f"{b:.4f}{stars(p)}", "t": round(t, 1),
                 "without controls x SUE": f"{b0:.4f}",
                 "Predicted": want, "Match": "yes" if (b > 0) == (want == "positive") else "no"})
print("\nSign check against the distraction hypothesis")
print(pd.DataFrame(_chk).to_string(index=False))

/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 677 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 677 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 691 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 700 singleton fixed effect(s) dropped from the model.
  warnings.warn(


Table B1. Attention channel: ATT = 11 - NRANK
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0721***     0.0213***     0.0651***     0.0315***
                                    (0.0014)      (0.0031)      (0.0013)      (0.0032)
ATT (11 - NRANK)                  -0.0009***        0.0002    -0.0009***        0.0001
                                    (0.0001)      (0.0003)      (0.0001)      (0.0003)
  SUE x ATT                        0.0019***      -0.0009*     0.0020***     -0.0011**
                                    (0.0002)      (0.0005)      (0.0002)      (0.0005)
--------------------------------------------------------------------------------------
Firm FE                                  Yes           Yes           Yes           Y

/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 677 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 677 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 691 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 700 singleton fixed effect(s) dropped from the model.
  warnings.warn(



Sign check against the distraction hypothesis
Window Returns SUE x ATT    t without controls x SUE Predicted Match
   ANN     C2C 0.0019***  8.4                 0.0015  positive   yes
 DRIFT     C2C  -0.0009* -1.8                -0.0009  negative   yes
   ANN     O2O 0.0020*** 10.0                 0.0017  positive   yes
 DRIFT     O2O -0.0011** -2.1                -0.0011  negative   yes


In [7]:
# 论文口径对照：NRANK 原始方向、FE 用 1-10 整数刻度、控制变量全部与 FE 交互、无公司 FE
d_att["sue_dec_int"] = d_att["sue_dec"].astype("float64")
d_att["sue_x_nrank"] = d_att["sue_dec_int"] * d_att["nrank"]
for c in CTRL:
    d_att[f"{c}_xFE"] = d_att[c].astype("float64") * d_att["sue_dec_int"]
for col in CARS.values():
    d_att[f"{col}_pct"] = d_att[col] * 100

XS_P = ["sue_dec_int", "nrank", "sue_x_nrank"] + CTRL + [f"{c}_xFE" for c in CTRL]
LAB_P = dict(LABELS, sue_dec_int="FE (earnings surprise decile 1-10)")

res_p = {k: run(f"{col}_pct", XS_P, fe="year + month + dow + ff10",
                data=d_att, cluster="date_id") for k, col in CARS.items()}
print(pub_table(res_p, ["sue_dec_int", "nrank", "sue_x_nrank"], LAB_P,
                title="Table B2. HLT (2009) Table III specification, replicated on 1996-2026",
                notes="Dependent variable: CAR in percentage points. FE is the earnings surprise decile (1-10),\n"
                      "NRANK the number-of-announcements decile. Standard errors clustered by announcement date.\n"
                      "No firm fixed effects and all controls interacted with FE, following the paper.\n"
                      "Published estimates (1995-2004, N = 112,839): FE x NRANK = -0.015 for CAR[0,1]\n"
                      "and +0.049 for CAR[2,61].",
                fe_rows=["Year / Month / DoW FE", "Industry FE (FF10)"]))

Table B2. HLT (2009) Table III specification, replicated on 1996-2026
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
FE (earnings surprise decile 1-10)     0.8165***     0.8636***     0.6986***     1.0378***
                                    (0.0519)      (0.1425)      (0.0457)      (0.1441)
NRANK                              0.1053***       -0.0047     0.1090***       -0.0120
                                    (0.0152)      (0.0419)      (0.0130)      (0.0437)
  FE x NRANK                      -0.0205***        0.0089    -0.0219***       0.0104*
                                    (0.0023)      (0.0060)      (0.0020)      (0.0063)
--------------------------------------------------------------------------------------
Year / Month / DoW FE                    Yes           Y

## 3.3 加入 LLM signal

**待接入**。ATT 的样本期覆盖 1996–2026，与现有 prediction 的 2004–2019 有充分交集，
这个 channel 可以先做起来。接入方案见 §4。

---

# 4. LLM signal 的接入方案

**来源**：`/project/dachxiu/yifei/news/experiment/US/ARTICLE/RidgeProximal/`
`QUESTION_CHOICE_pred_1d_is4cv3_cossim_0.8_O2O_RET_future_1d_O2O_RET_not_rank_normed_rolling_move_trading_days_expectation_only/`
（新闻级预测，2004–2019，只读）

**聚合**：新闻级 → firm-day → 事件窗口。先做两个以公告日 $d$ 为锚的窗口：

| 列名 | 窗口 | 含义 |
|---|---|---|
| `llm_ann` | $[d,\ d+1]$ | 与 $CAR^{ANN}$ 同窗口 |
| `llm_pre1` | $[d-1,\ d+1]$ | 多含公告前一天，捕捉盘前与提前泄露 |
| `llm_n_ann` `llm_n_pre1` | — | 各窗口内的新闻条数，用于判断覆盖度 |

窗口内没有新闻的事件记 `NaN`。列名统一加 `llm_` 前缀，之后要加别的窗口
（$[d-5,d]$、$[d,d+5]$ 等）直接追加同前缀的列。

**产出形式**：独立 side table `build/llm_signal.parquet`（键 `eid`），由 `build_llm_signal.py`
生成，在本 notebook 里按 `eid` 并入。先走 side table 是因为窗口与聚合方式还要反复试 ——
换一次不必重跑整条流水线。窗口定下来后再像 NRANK/ATT 一样固化进大表。

**接入后的完整方程**（两个 channel 通用，$M \in \{ADOPT,\ ATT\}$）：

$$
\begin{aligned}
CAR^{w}_{i,d}=\ &\alpha^{w}
+\beta^{w}_1 SUE^{rank}+\beta^{w}_2 M+\beta^{w}_3(SUE^{rank}\times M)
+\beta^{w}_4 LLM+\underbrace{\beta^{w}_5(M\times LLM)}_{\text{核心}}\\
&+\sum_k\gamma^{w}_k X_k+\sum_k\delta^{w}_k(X_k\times SUE^{rank})+\sum_k\phi^{w}_k(X_k\times LLM)
+\theta^{w}_i+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}
\end{aligned}
$$

比 §2.2 / §3.2 多出三组：$LLM$ 主效应、$M\times LLM$（核心）、以及 10 个 $X_k\times LLM$（系数 $\phi^{w}_k$）。